# Probe the training paradigms (V2: PPO)

Checks `core/rl/` with probe environments after Andy Jones,
[Debugging RL, Without the Agonizing Pain](https://andyljones.com/posts/rl-debugging.html):
tiny tasks whose correct answer is known exactly, each adding one piece of the
learning problem. Run in order, the first probe a paradigm fails names the broken
piece. The environments are in `core/rl/envs/probes.py`; the procedure (training,
scoring, longer budgets) is `core/rl/scripts/probe_check.py`, the same as
`python core/rl/scripts/probe_ppo.py` on the command line.

**How a probe is judged.** Not by the return, but by reading the trained networks
in fixed states: PPO's value `V` and the most likely action, against the exact
answer. A continuous output passes within 0.15 (values: 15 %, at least 0.15); a
categorical part must be exactly the expected category. In CTCE the group's value
is checked against the sum of the agents' values (its reward is the sum).

**Where it runs.** Here, or on the cluster in the Jupyter job (`hpc start`,
[notebook sessions](../../docs/hpc/notebooks.md)): on a SLURM job the runs share
one Ray instance and take one worker per physical core, capped by the thread
limit. The first cell shows where the kernel runs.

**How to use it.** Run the setup, choose in section b, train in section c (all
runs in one pool), read sections d to f, and use section g for longer budgets.
Runs are kept in `runs/probes_<preset>/` (not in git) and reused; `RETRAIN = True`
trains the selection again.

In [1]:
import os, socket, sys

print("host:", socket.gethostname())
print("SLURM job:", os.environ.get("SLURM_JOB_ID"), "CPUs:", os.environ.get("SLURM_CPUS_PER_TASK"))
print("python:", sys.executable)
print("cwd:", os.getcwd())

host: green21.hpclocal
SLURM job: 1047310 CPUs: 4
python: /gpfs/mariana/home/kerapo/miniforge3/envs/marlem/bin/python3.12
cwd: /gpfs/mariana/home/kerapo/marlem-v2


In [ ]:
from pathlib import Path

# The repository root, found from wherever the notebook runs (here or ~/marlem-v2 on
# the cluster), on sys.path; the training processes get it through PYTHONPATH.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "rl").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from core.rl.scripts import probe_check as pc

pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)
print(ROOT)

## a. The probes

| # | Probe | Action | Exact answer | Adds | If it is the first to fail |
|---|---|---|---|---|---|
| 1 | Constant | Box, ignored | V = 1 | value learning, reward, episode end | value loss, optimiser, reward delivery |
| 2 | Observation | Box, ignored | V(o) = o | observations reach the critic, per agent | the critic's input |
| 3 | Discount | Box, ignored | V(1) = 1, V(0) = γ | discounting over 2 steps | returns, episode ends |
| 4 | Action | Box | action = the agent's target | the policy update, per agent | the actor's update, action routing |
| 5 | Obs → action | Box | action = o/2 | the policy uses its own observation | the actor's input, batch alignment |
| 6 | Mixed action | category + level | c_i and t_i | the composite distribution | categorical + Gaussian parts |
| 7 | Gated action | category + gate | open at 0.5 with c_i | a gate decides whether the category counts (the market's no-order threshold) | choice learned too slowly: the policy stops "ordering" |
| 8 | Mutual choice | side + partner, pairs | mutual naming on the own sides, 1.1 | coordination (stage 1 of the clearing), with a safe DSO option | independent learners settle on the DSO (a stag hunt) |
| 9 | Conditional part | side + level | own side at its level, 1.05 | the best level depends on the side | the factorised policy's limit |
| 10 | Long episode | Box, ignored | V(t) = γ^(23−t) | the market's 24-step day | discounting and bootstrapping over a day |

Probes 4-9 name the agents so that their sorted order differs from their order
(probe 8 has four agents in two pairs). Probes 7 and 8 have a local optimum on
purpose, like the market (never order; trade with the DSO): a failure at the fast
budget says how hard the market's version is, not that the learner is broken.
Section g trains them longer.

## b. Choose what to test

- `PROBES`, `PARADIGMS`: any of 1-10 and `"ctce", "ctde", "dtde"`.
- `PRESET`: `"fast"` (2 × 64 network, short budgets: seconds to minutes per run) or
  `"thorough"` (RLlib's 2 × 256 network, learning rate 3e-4, twice the budget).
- `WORKERS`: runs trained at once; `None` chooses (cluster: one per physical core,
  capped by the thread limit; elsewhere half the CPUs).

In [ ]:
PROBES = list(pc.PROBES)  # 1-10
PARADIGMS = ["ctce", "ctde", "dtde"]
PRESET = "fast"
WORKERS = None
RETRAIN = False  # True: train the selection again, even where a run has finished
SEED = 0

budgets = {k: int(p.budget * pc.PRESETS[PRESET]["budget_scale"]) for k, p in pc.PROBES.items()}
print(len(PROBES) * len(PARADIGMS), "runs; budgets (environment steps):", budgets)
print("runs in", pc.runs_dir(PRESET))

## c. Train and score everything

One pool for every selected run; finished runs are only scored. On the cluster the
runs join one Ray instance started here (`parallel_setup`).

In [ ]:
summary, frames = pc.check(PROBES, PARADIGMS, PRESET, SEED, workers=WORKERS, retrain=RETRAIN)
pc.table(summary)

## d. One probe in detail

Every checked output of one probe: expected and got per paradigm. Change `PROBE`.

In [ ]:
def show(probe: int) -> None:
    found = {p: f for (k, p), f in frames.items() if k == probe}
    if not found:
        print("not run:", probe)
        return

    rows = summary[summary.probe == probe].set_index("paradigm")[["result", "worst", "budget"]]
    display(rows)
    expected = next(iter(found.values())).set_index(["state", "output"])[["expected"]]
    if len({len(f) for f in found.values()}) == 1 and "ctce" not in found:
        got = pd.DataFrame({p: f.set_index(["state", "output"]).got for p, f in found.items()})
        display(expected.join(got))
    else:  # CTCE checks the group's value, the others each agent's
        for paradigm, f in found.items():
            print(paradigm.upper())
            display(f.set_index(["state", "output"])[["expected", "got", "ok"]])


PROBE = 7
show(PROBE)

## e. The market probes, 6-10

Read with the market in mind:

- **7 Gated.** A failure with the gate's mean below 0 is the "never order" optimum:
  the category was not learned while the gate was open often enough.
- **8 Mutual choice.** Partners named but the DSO chosen means the safe option won;
  compare CTCE (one controller sees both agents) with DTDE and CTDE.
- **9 Conditional.** The level should match the side the agent chose; a level near 0
  is the compromise of an undecided side.
- **10 Long episode.** The values at hours 0, 12 and 23 against γ^(23−t).

In [ ]:
for probe in [p for p in (6, 7, 8, 9, 10) if p in PROBES]:
    print(f"\n=== {probe} {pc.PROBES[probe].name} ===")
    show(probe)

## f. Learner health

Per run: entropy at the start and end (it should fall as the policy sharpens, never
grow without limit), the last value loss and explained variance, and the
environment steps reached.

In [ ]:
def health(probe: int, paradigm: str) -> dict:
    run = pc.make_run(paradigm, probe, PRESET, SEED)
    row = {"status": pc.status(run)}
    path = Path(run.out_dir) / "metrics.csv"
    if path.exists():
        m = pd.read_csv(path)
        entropy = m.filter(like="/entropy").mean(axis=1).dropna()
        if len(entropy):
            row |= {"entropy start": entropy.iloc[0], "entropy end": entropy.iloc[-1]}
        for name in ("vf_loss", "vf_explained_var"):
            column = m.filter(like=f"modules/{name}_mean")
            if column.shape[1]:
                row[name] = column.iloc[:, 0].dropna().iloc[-1]
        row["env steps"] = int(m.env_steps.iloc[-1])
    return row


pd.DataFrame({(k, p.upper()): health(k, p) for k in PROBES for p in PARADIGMS}).T

## g. Longer budgets for what failed

`escalate` trains every failing combination again at each larger budget in turn
(here 2×, then 4×, then 8× the preset's), all failures of one scale in one pool,
until they pass. A slow learner passes at some scale; a broken one does not get
closer. The table gives the smallest scale each combination passed at (`NaN`:
none). On the cluster this is where the extra cores pay off.

In [ ]:
SCALES = [2, 4, 8]
passed = pc.escalate(summary, SCALES, PRESET, SEED, workers=WORKERS)
passed.pivot(index=["probe", "name"], columns="paradigm", values="passed at")